## 1. Loading and Exploring the Dataset

First, I will load the SMS Spam Collection dataset using Pandas. The dataset contains text messages labeled as either `ham` (legitimate messages) or `spam` (unwanted messages).

I will inspect the first few records, dataset dimensions, and the distribution of the two labels.

In [101]:
import pandas as pd

# Load the dataset
df = pd.read_csv(
    "data/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

# Display the first five messages
display(df.head())

# Check the dataset dimensions
print("Dataset shape:", df.shape)

# Count ham and spam messages
print("\nLabel distribution:")
print(df["label"].value_counts())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


## 2. Separating Features and Labels

The dataset contains two columns:
- `message`: The actual SMS text that the model will learn from.
- `label`: The correct classification (`ham` or `spam`).

I will separate these columns into features (X) and labels (y) so they can be used to train and evaluate the model.

In [102]:
# Separate messages and labels
emails = df["message"]
labels = df["label"]

# Check the results
print("Messages:")
print(emails.head())

print("\nLabels:")
print(labels.head())

Messages:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: str

Labels:
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: str


## 3. Splitting the Dataset

I will divide the dataset into training and testing sets using Scikit-learn's `train_test_split`.

The training set will contain 80% of the messages, while the testing set will contain the remaining 20%. I will use `random_state=42` to make the split reproducible.

In [103]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    emails,
    labels,
    test_size=0.2,
    random_state=42
)

# Display the sizes of each set
print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

print("\nTraining labels:", len(y_train))
print("Testing labels:", len(y_test))

Training messages: 4457
Testing messages: 1115

Training labels: 4457
Testing labels: 1115


## 4. Converting Text into Numerical Features

Machine learning models work with numerical data rather than raw text. I will use `TfidfVectorizer` to convert SMS messages into numerical features.

TF-IDF assigns importance scores to words based on their frequency in a message and how common they are across the collection of messages.

In [104]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vec = TfidfVectorizer()

# Learn vocabulary and transform training messages
X_train_tfidf = vec.fit_transform(X_train)

# Transform testing messages using the same vocabulary
X_test_tfidf = vec.transform(X_test)

# Display the resulting shapes
print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (4457, 7702)
Testing data shape: (1115, 7702)


## 5. Training the Spam Classifier

I will train a Multinomial Naive Bayes classifier using the TF-IDF features extracted from the training messages.

The model will learn patterns that help distinguish spam messages from legitimate messages (`ham`).

In [105]:
from sklearn.naive_bayes import MultinomialNB

# Create the Naive Bayes classifier
model = MultinomialNB()

# Train the model using the training data
model.fit(X_train_tfidf, y_train)

print("Spam classifier trained successfully!")

Spam classifier trained successfully!


## 6. Evaluating Model Accuracy

I will evaluate the trained classifier using the testing dataset. The model will predict whether each message is spam or ham, and I will compare its predictions with the actual labels using accuracy score.

Accuracy measures the proportion of messages classified correctly.

In [106]:
from sklearn.metrics import accuracy_score

# Predict labels for the testing messages
predictions = model.predict(X_test_tfidf)

# Calculate the model's accuracy
accuracy = accuracy_score(y_test, predictions)

print(f"Model accuracy: {accuracy:.2%}")

Model accuracy: 96.68%


## 7. Testing the Model with Tricky Messages

I will test the classifier using five carefully designed SMS messages to investigate its weaknesses.

The messages will use techniques such as unusual spelling, negation, professional-sounding language, and misleading context. I will compare the model's predictions with the expected labels to identify correct classifications and mistakes.

In [107]:
# Five messages designed to test the model's weaknesses
tricky_messages = [
    "Congratulations! Claim your fr33 prize now by calling 08001234567.",
    "You have NOT won any prize. This is only a reminder about our meeting.",
    "Dear customer, kindly verify your account details to receive your reward.",
    "Hey, can you send me the free notes from today's class?",
    "Your free subscription expires tomorrow. Please renew it through the official university portal."
]

# Expected labels for our five messages
expected_labels = [
    "spam",
    "ham",
    "spam",
    "ham",
    "ham"
]

print("Number of test messages:", len(tricky_messages))
print("Number of expected labels:", len(expected_labels))

Number of test messages: 5
Number of expected labels: 5


## 8. Testing the Tricky Messages

I will use the trained classifier to predict whether each tricky message is spam or ham. I will then compare its predictions with my expected labels to identify any misclassifications.

In [108]:
# Convert the tricky messages into TF-IDF features
tricky_tfidf = vec.transform(tricky_messages)

# Predict whether each message is spam or ham
tricky_predictions = model.predict(tricky_tfidf)

# Display the predictions alongside the expected labels
for i, (message, prediction, expected) in enumerate(
    zip(tricky_messages, tricky_predictions, expected_labels),
    start=1
):
    result = "Correct" if prediction == expected else "Misclassified"

    print(f"\nMessage {i}: {message}")
    print(f"Predicted: {prediction}")
    print(f"Expected:  {expected}")
    print(f"Result:    {result}")


Message 1: Congratulations! Claim your fr33 prize now by calling 08001234567.
Predicted: spam
Expected:  spam
Result:    Correct

Message 2: You have NOT won any prize. This is only a reminder about our meeting.
Predicted: ham
Expected:  ham
Result:    Correct

Message 3: Dear customer, kindly verify your account details to receive your reward.
Predicted: ham
Expected:  spam
Result:    Misclassified

Message 4: Hey, can you send me the free notes from today's class?
Predicted: ham
Expected:  ham
Result:    Correct

Message 5: Your free subscription expires tomorrow. Please renew it through the official university portal.
Predicted: ham
Expected:  ham
Result:    Correct


## 9. Analyzing the Results

I will compare the model's predictions with the expected labels for all five tricky messages.

This analysis will help identify the model's strengths and weaknesses, including cases where it correctly identifies spam and legitimate messages and cases where it makes incorrect predictions.

In [109]:
# Create a DataFrame to organize the experiment results
results_df = pd.DataFrame({
    "Message": tricky_messages,
    "Predicted Label": tricky_predictions,
    "Expected Label": expected_labels
})

# Check whether each prediction matches the expected label
results_df["Result"] = results_df.apply(
    lambda row: "Correct"
    if row["Predicted Label"] == row["Expected Label"]
    else "Misclassified",
    axis=1
)

# Display the results
display(results_df)

# Calculate accuracy on the tricky messages
tricky_accuracy = accuracy_score(expected_labels, tricky_predictions)

print(f"Tricky message accuracy: {tricky_accuracy:.2%}")

,Message,Predicted Label,Expected Label,Result
0,Congratulations! Claim your fr33 prize now by ...,spam,spam,Correct
1,You have NOT won any prize. This is only a rem...,ham,ham,Correct
2,"Dear customer, kindly verify your account deta...",ham,spam,Misclassified
3,"Hey, can you send me the free notes from today...",ham,ham,Correct
4,Your free subscription expires tomorrow. Pleas...,ham,ham,Correct


Tricky message accuracy: 80.00%


### Analysis of the Results

The classifier correctly identified four out of the five tricky messages, achieving 80% accuracy in this experiment.

- **Message 1:** The model correctly identified the spam message despite the unusual spelling "fr33". Other familiar spam-related words may have helped the classifier.
- **Message 2:** The model correctly classified the message as ham, even though it contained the words "won" and "prize".
- **Message 3:** The model incorrectly classified a suspicious account-verification message as ham. Its polite and professional wording may have made it resemble legitimate messages.
- **Message 4:** The model correctly identified a legitimate message about class notes, despite the use of the word "free".
- **Message 5:** The model correctly classified a legitimate university subscription reminder as ham.

The main weakness observed was the model's inability to identify the suspicious account-verification message as spam.

## 10. Conclusion

The Naive Bayes spam classifier achieved 96.68% accuracy on the original test dataset and correctly classified four out of five specially designed messages. This experiment taught me that even a model with high accuracy can make mistakes when spam messages use language that resembles legitimate communication.